# Train and Segment a Tiled Area

Train on a labeled area, prepare larger inference rasters as overlapping tiles, and segment the tiles with the trained bundle.


## Inputs

Supply your own training imagery, canopy height, first-return density, and crown polygons, plus imagery and lidar rasters for inference.
The example paths below are placeholders; these data files are not included in the repository.
See [Data Preparation](../data_preparation/) for raster requirements and [Training](../training/#prepare-crown-annotations) for crown annotations.
The required base checkpoint is linked in [Training](../training/#base-checkpoint).

Run this notebook in an environment where samstars is installed. It can run outside a repository checkout.


In [ ]:
from pathlib import Path

from samstars import train, segment
from samstars.data import build_tiles


## Set Paths and Image Bands

Use the same band content and cost settings for training and inference tiles. The example assumes both source images have the same band order.


In [ ]:
training_dir = Path("data/training").resolve()
inference_dir = Path("data/inference").resolve()
sam_base_ckpt = Path("models/sam_vit_l_0b3195.pth").resolve()
model_dir = Path("models/crowns").resolve()

rgb_bands = (1, 2, 3)  # One-based red, green, blue indices in the source images.
nir_band = None        # Set to a near-infrared band index when available.
device = "cpu"

tiles_dir = Path("prepared/tiles").resolve()
inference_run_dir = Path("runs/tiled_segmentation").resolve()


## Train a Model Bundle

The reduced epoch counts provide an initial run for inspection; assess training settings on held-out areas.


In [ ]:
model = train(
    img=training_dir / "image.tif",
    chm=training_dir / "chm.tif",
    den=training_dir / "first_return_density.tif",
    crowns=training_dir / "crowns.gpkg",
    sam_base_ckpt=sam_base_ckpt,
    out=model_dir,
    rgb_bands=rgb_bands,
    nir_band=nir_band,
    sam_epochs=3,
    stardist_epochs=10,
    storage="persistent",
    device=device,
)

print(model.root)


## Build Inference Tiles

With a projected CRS in meters, these settings create approximately 500-meter tiles with 20 meters of overlap.


In [ ]:
build_tiles(
    img=inference_dir / "image.tif",
    chm=inference_dir / "chm.tif",
    den=inference_dir / "first_return_density.tif",
    out=tiles_dir,
    rgb_bands=rgb_bands,
    nir_band=nir_band,
    tile=500.0,
    overlap=20.0,
)


## Segment Prepared Tiles

The example enables duplicate suppression at an intersection-over-union threshold of 0.6. Inspect tile boundaries when selecting a threshold; see [Large Raster Runs](../large_raster_runs/#overlap-and-duplicate-crowns).


In [ ]:
result = segment(
    model=model,
    tiles_dir=tiles_dir,
    out=inference_run_dir,
    storage="persistent",
    device=device,
    stardist_dedup_iou_thr=0.6,
)

print(result.crowns)


## Inspect Crowns

Open the combined crown GeoPackage in GIS software, or read it with GeoPandas.


In [ ]:
import geopandas as gpd

crowns = gpd.read_file(result.crowns, layer="crowns")
crowns.head()


## Optional Tuning

The defaults are suitable for checking the workflow on a small area. The settings below can be added to the training or segmentation calls when needed.

| Setting | Purpose |
| --- | --- |
| Training epochs | Increase sam_epochs and stardist_epochs to extend training, assessing results on held-out areas. |
| Training chips | Set chip_size and chip_stride in pixels to control the training windows. |
| Seed detection | Set seed_d_min, seed_min_dist_px, and seed_min_canopy_height during training or tile preparation; see [Data Preparation](../data_preparation/#create-seed-points). |
| Texture band | Set texture_band during training to use a specific source band instead of mean RGB intensity. Use the same setting for later tile preparation. |
| Refinement thresholds | Pass stardist_prob_thr and stardist_nms_thr together to segmentation, or omit both to use model thresholds. |

See the [API reference](../api/top_level/) for parameter details and [Installation](../installation/) for accelerator setup.

For large tiles, stardist_window_size_m and stardist_window_overlap_m divide refinement into smaller windows. See [Advanced Workflows](../advanced_workflows/) for resuming persistent runs.
